## 1. import

In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors

from catboost import CatBoostClassifier, Pool

## 2. config

In [2]:
class CFG:
    SEED = 42
    N_SPLITS = 5
    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    ORIG_PATH = "/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv"

    N_NEIGHBORS = 32
    SNAP_GROUP_COLS = ["Season", "Irrigation_Type", "Region"]
    SNAP_NUM_COLS = [
        "Soil_Moisture",
        "Temperature_C",
        "Humidity",
        "Rainfall_mm",
        "Previous_Irrigation_mm",
    ]

    MODEL_PARAMS = {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "iterations": 3000,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 5.0,
        "random_strength": 1.0,
        "bagging_temperature": 0.0,
        "grow_policy": "SymmetricTree",
        "bootstrap_type": "Bayesian",
        "random_seed": SEED,
        "verbose": 200,
        "early_stopping_rounds": 200,
        "task_type": "GPU" if torch.cuda.is_available() else "CPU",
    }

## 3. column groups

In [3]:
# =========================
# Column groups
# =========================
NUM_COLS_BASE = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Temperature_C",
    "Humidity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]

CAT_COLS_BASE = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Region",
]

BIN_COLS_BASE = [
    "Mulching_Used",
]

CAT_ALL_BASE = CAT_COLS_BASE + BIN_COLS_BASE
FEATURE_COLS_BASE = NUM_COLS_BASE + CAT_COLS_BASE + BIN_COLS_BASE

## 4. utility

In [4]:
# =========================
# Utilities
# =========================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def cast_dtypes(train_df, test_df, orig_df):
    for col in CAT_ALL_BASE:
        train_df[col] = train_df[col].astype(str)
        test_df[col] = test_df[col].astype(str)
        orig_df[col] = orig_df[col].astype(str)

    for col in NUM_COLS_BASE:
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
        test_df[col] = pd.to_numeric(test_df[col], errors="coerce")
        orig_df[col] = pd.to_numeric(orig_df[col], errors="coerce")

    return train_df, test_df, orig_df

## 5. load data

In [5]:
seed_everything(CFG.SEED)

train_df = pd.read_csv(CFG.TRAIN_PATH)
test_df = pd.read_csv(CFG.TEST_PATH)
orig_df = pd.read_csv(CFG.ORIG_PATH)

train_df, test_df, orig_df = cast_dtypes(train_df, test_df, orig_df)

print(train_df.shape, test_df.shape, orig_df.shape)

(630000, 21) (270000, 20) (10000, 20)


## 6. label encode

In [6]:
le = LabelEncoder()
y = le.fit_transform(train_df[CFG.TARGET_COL])
class_names = list(le.classes_)
n_classes = len(class_names)

orig_y = le.transform(orig_df[CFG.TARGET_COL])

global_prior = np.bincount(orig_y, minlength=n_classes).astype(np.float64)
global_prior = global_prior / global_prior.sum()

print("classes:", class_names)
print("global_prior:", global_prior)

classes: ['High', 'Low', 'Medium']
global_prior: [0.0336 0.5864 0.38  ]


## 7. snap feature generator

In [7]:
def build_snap_features(
    query_df: pd.DataFrame,
    orig_df: pd.DataFrame,
    orig_y: np.ndarray,
    group_cols,
    num_cols,
    n_neighbors=32,
):
    """
    query_df の各行について、同じ group_cols を持つ original rows の中から
    num_cols 空間で近い近傍を取り、その class prior と距離特徴を作る。
    """
    out = pd.DataFrame(index=query_df.index)

    for c in class_names:
        out[f"snap_prior__{c}"] = np.nan
    out["snap_dist_mean"] = np.nan
    out["snap_dist_min"] = np.nan
    out["snap_group_count"] = np.nan

    # group key 作成
    q_key = query_df[group_cols].astype(str).agg("||".join, axis=1)
    o_key = orig_df[group_cols].astype(str).agg("||".join, axis=1)

    orig_work = orig_df.copy()
    orig_work["_group_key"] = o_key
    orig_work["_y"] = orig_y

    query_work = query_df.copy()
    query_work["_group_key"] = q_key

    for gval, q_idx in query_work.groupby("_group_key").groups.items():
        q_part = query_work.loc[q_idx, num_cols].copy()
        o_part = orig_work.loc[orig_work["_group_key"] == gval, num_cols].copy()
        y_part = orig_work.loc[orig_work["_group_key"] == gval, "_y"].to_numpy()

        if len(o_part) == 0:
            # unseen group
            for i, c in enumerate(class_names):
                out.loc[q_idx, f"snap_prior__{c}"] = global_prior[i]
            out.loc[q_idx, "snap_dist_mean"] = 999.0
            out.loc[q_idx, "snap_dist_min"] = 999.0
            out.loc[q_idx, "snap_group_count"] = 0
            continue

        # 欠損埋め
        med = o_part.median()
        q_part = q_part.fillna(med)
        o_part = o_part.fillna(med)

        # 標準化
        scaler = StandardScaler()
        o_scaled = scaler.fit_transform(o_part)
        q_scaled = scaler.transform(q_part)

        k = min(n_neighbors, len(o_part))
        nn = NearestNeighbors(n_neighbors=k, metric="euclidean")
        nn.fit(o_scaled)

        dists, inds = nn.kneighbors(q_scaled)

        snap_priors = np.zeros((len(q_idx), n_classes), dtype=np.float64)
        for row_i in range(len(q_idx)):
            neigh_y = y_part[inds[row_i]]
            counts = np.bincount(neigh_y, minlength=n_classes).astype(np.float64)
            snap_priors[row_i] = counts / counts.sum()

        for i, c in enumerate(class_names):
            out.loc[q_idx, f"snap_prior__{c}"] = snap_priors[:, i]

        out.loc[q_idx, "snap_dist_mean"] = dists.mean(axis=1)
        out.loc[q_idx, "snap_dist_min"] = dists.min(axis=1)
        out.loc[q_idx, "snap_group_count"] = len(o_part)

    return out

## 8. make snap features

In [8]:
train_snap = build_snap_features(
    query_df=train_df,
    orig_df=orig_df,
    orig_y=orig_y,
    group_cols=CFG.SNAP_GROUP_COLS,
    num_cols=CFG.SNAP_NUM_COLS,
    n_neighbors=CFG.N_NEIGHBORS,
)

test_snap = build_snap_features(
    query_df=test_df,
    orig_df=orig_df,
    orig_y=orig_y,
    group_cols=CFG.SNAP_GROUP_COLS,
    num_cols=CFG.SNAP_NUM_COLS,
    n_neighbors=CFG.N_NEIGHBORS,
)

print(train_snap.shape, test_snap.shape)
train_snap.head()

(630000, 6) (270000, 6)


,snap_prior__High,snap_prior__Low,snap_prior__Medium,snap_dist_mean,snap_dist_min,snap_group_count
0,0.00000,0.56250,0.43750,1.902572,1.015929,162.0
1,0.00000,0.75000,0.25000,1.502747,0.935671,174.0
2,0.03125,0.68750,0.28125,2.000974,0.643223,160.0
3,0.03125,0.53125,0.43750,1.776222,0.897852,187.0
4,0.00000,0.81250,0.18750,2.064296,0.706717,170.0


## 9. merge snap features

In [9]:
SNAP_COLS = list(train_snap.columns)

train_df = pd.concat([train_df.reset_index(drop=True), train_snap.reset_index(drop=True)], axis=1)
test_df = pd.concat([test_df.reset_index(drop=True), test_snap.reset_index(drop=True)], axis=1)

FEATURE_COLS_SNAP = FEATURE_COLS_BASE + SNAP_COLS

print("SNAP_COLS:", SNAP_COLS)
print("n SNAP_COLS:", len(SNAP_COLS))

SNAP_COLS: ['snap_prior__High', 'snap_prior__Low', 'snap_prior__Medium', 'snap_dist_mean', 'snap_dist_min', 'snap_group_count']
n SNAP_COLS: 6


## 10. CV

In [10]:
X = train_df[FEATURE_COLS_SNAP].copy()
X_test = test_df[FEATURE_COLS_SNAP].copy()

skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

oof_proba = np.zeros((len(train_df), n_classes), dtype=np.float32)
test_proba = np.zeros((len(test_df), n_classes), dtype=np.float32)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    print("=" * 60)
    print(f"fold {fold} / {CFG.N_SPLITS}")

    X_tr = X.iloc[tr_idx].copy()
    X_va = X.iloc[va_idx].copy()
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    train_pool = Pool(
        data=X_tr,
        label=y_tr,
        cat_features=CAT_ALL_BASE,
    )
    valid_pool = Pool(
        data=X_va,
        label=y_va,
        cat_features=CAT_ALL_BASE,
    )
    test_pool = Pool(
        data=X_test,
        cat_features=CAT_ALL_BASE,
    )

    model = CatBoostClassifier(**CFG.MODEL_PARAMS)

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
    )

    va_proba = model.predict_proba(valid_pool)
    te_proba = model.predict_proba(test_pool)

    oof_proba[va_idx] = va_proba
    test_proba += te_proba / CFG.N_SPLITS

    va_pred = np.argmax(va_proba, axis=1)
    fold_bacc = balanced_accuracy_score(y_va, va_pred)
    fold_scores.append(fold_bacc)

    print(f"fold {fold} balanced_accuracy = {fold_bacc:.6f}")

fold 1 / 5
0:	learn: 1.0374401	test: 1.0373987	best: 1.0373987 (0)	total: 127ms	remaining: 6m 21s
200:	learn: 0.0644274	test: 0.0646963	best: 0.0646963 (200)	total: 3s	remaining: 41.9s
400:	learn: 0.0630550	test: 0.0636688	best: 0.0636687 (399)	total: 5.37s	remaining: 34.8s
600:	learn: 0.0618129	test: 0.0629049	best: 0.0629049 (600)	total: 7.79s	remaining: 31.1s
800:	learn: 0.0604632	test: 0.0622498	best: 0.0622497 (799)	total: 10.2s	remaining: 28.1s
1000:	learn: 0.0594749	test: 0.0617848	best: 0.0617846 (997)	total: 12.9s	remaining: 25.8s
1200:	learn: 0.0585679	test: 0.0614208	best: 0.0614208 (1200)	total: 15.7s	remaining: 23.6s
1400:	learn: 0.0578168	test: 0.0611431	best: 0.0611431 (1400)	total: 18.5s	remaining: 21.2s
1600:	learn: 0.0568727	test: 0.0608304	best: 0.0608304 (1600)	total: 21.4s	remaining: 18.7s
1800:	learn: 0.0560691	test: 0.0606283	best: 0.0606283 (1800)	total: 23.9s	remaining: 15.9s
2000:	learn: 0.0552047	test: 0.0604068	best: 0.0604068 (2000)	total: 26.5s	remaining: 

## 11. summary

In [11]:
print("=" * 60)
print("fold scores:", [round(s, 6) for s in fold_scores])
print(f"cv mean balanced_accuracy = {np.mean(fold_scores):.6f}")
print(f"cv std  balanced_accuracy = {np.std(fold_scores):.6f}")

oof_pred = np.argmax(oof_proba, axis=1)
oof_bacc = balanced_accuracy_score(y, oof_pred)
print(f"OOF balanced_accuracy = {oof_bacc:.6f}")

print("\nPer-class recall on OOF:")
for cls_idx, cls_name in enumerate(class_names):
    mask = (y == cls_idx)
    recall = (oof_pred[mask] == y[mask]).mean()
    print(f"{cls_name}: {recall:.6f}")

fold scores: [np.float64(0.957154), np.float64(0.961422), np.float64(0.959516), np.float64(0.958844), np.float64(0.957529)]
cv mean balanced_accuracy = 0.958893
cv std  balanced_accuracy = 0.001528
OOF balanced_accuracy = 0.958893

Per-class recall on OOF:
High: 0.908896
Low: 0.994277
Medium: 0.973506


## 12. save

In [12]:
test_pred = np.argmax(test_proba, axis=1)
test_label = le.inverse_transform(test_pred)

sub_df = pd.DataFrame({
    CFG.ID_COL: test_df[CFG.ID_COL],
    CFG.TARGET_COL: test_label
})

np.save("oof_catboost_snap_min_proba.npy", oof_proba)
np.save("pred_catboost_snap_min_proba.npy", test_proba)
sub_df.to_csv("submission_catboost_snap_min.csv", index=False)

print("\nSaved:")
print("- oof_catboost_snap_min_proba.npy")
print("- pred_catboost_snap_min_proba.npy")
print("- submission_catboost_snap_min.csv")


Saved:
- oof_catboost_snap_min_proba.npy
- pred_catboost_snap_min_proba.npy
- submission_catboost_snap_min.csv
